# 05. Reranking (재순위화): AI 트렌드 보고서 기반

## 개념

벡터 검색은 빠르지만 **질문과 문서의 세밀한 관련성 판단이 부족**할 수 있다.  
Reranker는 벡터 검색으로 넓게 가져온 후보 문서를 **질문과의 관련성 기준으로 다시 정렬**한다.

이 실습에서는 앞에서 생성한 **AI 트렌드 보고서 Markdown 기반 ChromaDB**를 사용한다.

```text
벡터 검색 (Bi-Encoder)          Reranking (Cross-Encoder)
────────────────────────         ──────────────────────────
질문 → 벡터                      질문 + 문서를 동시에 입력
문서 → 벡터           →          관련성 점수 계산
거리 기반 검색                    정밀 재정렬
속도 빠름                         속도 느림, 정확도 높음
```

## 2단계 검색 전략

1. **1단계**: 벡터 검색으로 후보 문서를 넓게 확보한다. 예: `k=10`
2. **2단계**: Reranker로 질문과 더 관련 있는 상위 문서만 선별한다. 예: `top_n=3`

> 실행 전 확인: `02_data_pdf_to_md.ipynb`를 먼저 실행하여 `ai_trend_report_2026_md` ChromaDB가 생성되어 있어야 한다.


## Markdown 기반 VectorDB 불러오기

이번 실습에서는 02번에서 생성한 Markdown 기반 ChromaDB를 사용한다.  
Reranking은 새로운 VectorDB를 만드는 과정이 아니라, 이미 검색된 후보 문서의 순서를 다시 조정하는 과정이다.

따라서 먼저 기존 VectorDB에서 후보 문서를 넓게 검색하고, 이후 Reranker를 적용하여 질문과 더 직접적으로 관련 있는 문서를 상위로 올린다.

In [1]:
from pathlib import Path

from dotenv import load_dotenv
load_dotenv(override=True, dotenv_path="../.env")

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 앞 실습(02_data_pdf_to_md.ipynb)에서 생성한 AI 트렌드 보고서 Markdown 기반 ChromaDB 불러오기
DB_PATH = Path("chroma_db/ai_trend_report_md")
COLLECTION_NAME = "ai_trend_report_2026_md"
vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=DB_PATH
)

print("AI 트렌드 보고서 Vector Store 준비 완료")
print(f"저장된 문서 chunk 수: {vector_store._collection.count()}")


AI 트렌드 보고서 Vector Store 준비 완료
저장된 문서 chunk 수: 29


## FlashrankRerank 사용: 로컬 실행 기반 Reranking

`FlashRank`는 경량 Cross-Encoder 기반 Reranker이다. API 키 없이 로컬에서 동작한다.

설치가 필요하면 아래 명령을 먼저 실행한다.

```bash
uv add flashrank langchain-community langchain-classic
```

주의: FlashrankRerank는 로컬에서 간단히 사용할 수 있다는 장점이 있지만, 사용하는 모델에 따라 한국어 문서의 관련성 판단 성능이 제한될 수 있다.  
따라서 본 실습에서는 Reranking의 동작 원리를 이해하는 데 초점을 둔다.

### 왜 ContextualCompressionRetriever를 사용하는가?

LangChain에서는 검색된 문서를 후처리하는 과정을 `Contextual Compression` 흐름으로 다룬다.  
여기서 Compression은 반드시 문장을 요약하거나 줄인다는 뜻만은 아니다.

Reranker를 사용할 때도 다음 흐름이 적용된다.

1. 기본 Retriever가 후보 문서를 넓게 검색한다.
2. Compressor 또는 Reranker가 후보 문서를 다시 평가한다.
3. 관련성이 높은 문서만 남긴다.

따라서 이 실습에서는 `ContextualCompressionRetriever`를 Reranking 결과를 반환하는 래퍼로 사용한다.

### k와 top_n의 차이

`k=10`은 VectorDB에서 먼저 가져올 후보 문서 수이다.  
즉, Reranker가 평가할 후보군의 크기이다.

`top_n=3`은 Reranker가 후보 문서 중 최종적으로 남길 문서 수이다.

따라서 이 실습의 검색 흐름은 다음과 같다.

```text
질문
→ 벡터 검색으로 후보 문서 10개 검색
→ Reranker가 10개 문서를 다시 평가
→ 관련성 높은 상위 3개 문서만 반환
```

In [2]:
from langchain_community.document_compressors import FlashrankRerank
from langchain_classic.retrievers import ContextualCompressionRetriever


def doc_label(doc):
    """문서 출력용 metadata 정보를 반환한다."""
    page = doc.metadata.get("page", None)
    source = doc.metadata.get("source", "?")
    chunk_id = doc.metadata.get("chunk_id", "?")
    doc_type = doc.metadata.get("doc_type", "?")
    source_name = doc.metadata.get("source_name", "?")

    if isinstance(page, int):
        return f"page={page + 1} | chunk_id={chunk_id} | source={source}"

    return f"chunk_id={chunk_id} | doc_type={doc_type} | source_name={source_name}"


# 1단계: 벡터 검색으로 후보 문서를 넓게 검색한다.
base_retriever = vector_store.as_retriever(search_kwargs={"k": 10})

# 2단계: Reranker로 상위 3개 문서를 재선별한다.
reranker = FlashrankRerank(top_n=3)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=base_retriever
)

question = "AI 산업, AI 기술, AI 정책의 핵심 이슈를 비교해줘."
results = compression_retriever.invoke(question)

print("[Reranking 적용 후]")
print("relevance_score: 높을수록 질문과 더 관련 있음")
print(f"Reranking 후 문서 수: {len(results)}")
for i, doc in enumerate(results, start=1):
    score = doc.metadata.get("relevance_score", "N/A")
    if isinstance(score, float):
        score_text = f"{score:.4f}"
    else:
        score_text = score

    print(f"\n[{i}] 관련성 점수: {score} | source: {doc_label(doc)}")
    print(doc.page_content[:400])


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[Reranking 적용 후]
relevance_score: 높을수록 질문과 더 관련 있음
Reranking 후 문서 수: 3

[1] 관련성 점수: 0.9996312260627747 | source: chunk_id=7 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md
- - 생성형 AI 고도화, AI 인프라 경쟁, 안전·책임성 강화, 국제 기준 정비 등 주요 변화 요인을 식별하여 산업·기술·정책 분야의 변화 방향성을 명확히 도출함 

- 더 나아가 분석 결과를 기반으로 국가 정책, 기업 전략, 기술 개발 방향 설정에 활용 가능한 실질적 인사이트를 제공하고, 중장기 전략 수립을 위한 근거를 제공하고자 함 

- - 특히 정책 담당자에게는 AI 기본법·규제체계 설계 등 법적 기반 마련 시 고려해야 할 우선순위를, 기업에는 투자·사업 포트폴리오 조정 방향을, 연구기관에는 후속 정량·정성 연구가 필요한 공백 영역을 제시하는 것을 목표로 함 

- 산업·기술·정책 세부 분석을 통해 다음과 같은 분야별 기대효과를 도출하고자 함 

- - 산업 분야에서는 AI 확산이 산업 구조·시장

[2] 관련성 점수: 0.9996051788330078 | source: chunk_id=24 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md
- - EU AI Act 등 글로벌 규제와의 정합성을 높이기 위해 국내 AI 기본법의 시행령·가이드라인이 구체화되고, 수출 기업을 위한 규제 대응·인증 지원이 확대될 전망임 

- - 의료·채용 등 고위험 AI의 안전성 검증과 제3자 인증이 필수화되고, 생성형 AI 부작용 대응을 위한 워터마크·딥페이크 탐지 기술이 법제화될 것으로 예상됨 

- – – 

- ※ EU AI 규제는 금지 고위험 제한적 위험 최소 위험으로 구분하는 글로벌 표준 모델[5)] 

## **[ 그림 6 ] AI 규제 기준과 안전성 확보 기술의 대표 사례** 


- k가 너무 작으면 좋은 문서가 후보에 포함되지 않을 수 있다.
- 반대로 k가 너무 크면 Reranking 시간이 길어지고 비용이 증가할 수 있다.

### relevance_score 해석

`relevance_score`는 Reranker가 질문과 문서의 관련성을 평가한 점수이다.  
일반적으로 점수가 높을수록 질문과 더 관련 있는 문서로 해석한다.

다만 이 점수는 모델마다 스케일이 다를 수 있으므로, 서로 다른 Reranker 모델의 점수를 직접 비교하면 안 된다.  
이 실습에서는 같은 Reranker 안에서 문서의 상대적 순위를 비교하는 용도로 사용한다.

### Flashrank Reranking과 LLM Reranking의 차이

Flashrank Reranking은 전용 Reranker 모델이 질문과 문서 쌍을 입력받아 관련성 점수를 계산하는 방식이다.  
상대적으로 빠르고 비용이 낮지만, 모델의 언어 처리 성능과 도메인 적합성에 영향을 받는다.

LLM Reranking은 GPT 같은 LLM에게 문서 관련성을 직접 평가하게 하는 방식이다.  
도메인 문맥을 유연하게 판단할 수 있지만, 후보 문서 수만큼 호출이 늘어나므로 비용과 시간이 증가한다.

수업에서는 Flashrank를 Reranking의 기본 구조 이해용으로, LLM Reranking을 Reranking 원리를 직관적으로 이해하기 위한 비교 방식으로 사용한다.

### Reranking 전후 비교 기준

Reranking 전후를 비교할 때는 단순히 문서 순서만 보지 않는다.  
다음 기준으로 확인한다.

1. 질문에 직접 답할 수 있는 문서가 상위로 올라왔는가?
2. 관련성이 낮은 문서가 뒤로 밀렸는가?
3. 같은 내용의 중복 문서가 줄었는가?
4. 표, 목록, 특정 섹션 정보가 필요한 질문에서 적절한 청크가 선택되었는가?
5. 최종 LLM 답변에 불필요한 context가 줄었는가?

## LLM 기반 Reranking (API만으로 동작)

FlashRank 같은 별도 모델을 사용하지 않고, LLM에게 각 후보 문서의 관련성 점수를 직접 평가하게 할 수도 있다.

이 방식은 구현이 단순하지만 후보 문서 수만큼 LLM 호출이 발생하므로 비용과 시간이 증가한다.


### LLM Reranking 사용 시 주의점

LLM에게 숫자만 출력하라고 지시해도 항상 정수만 반환된다고 보장할 수는 없다.  
예를 들어 `8점`, `Score: 8`, `관련성 8`처럼 출력될 수 있다.

현재 실습에서는 예외가 발생하면 0점으로 처리한다.  
실무에서는 정규표현식으로 숫자만 추출하거나, JSON 형식 출력 또는 structured output을 사용하는 것이 더 안정적이다.

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document

rerank_prompt = ChatPromptTemplate.from_messages([
    ("system", """AI 트렌드 보고서 문서와 질문의 관련성을 0~10 사이의 정수로 평가하세요.
숫자만 출력하세요.

평가기준:
- 질문에 직접 답할 수 있는 문서이면 높은 점수
- 질문과 일부 관련만 있으면 중간 점수
- 질문과 관련이 거의 없으면 낮은 점수"""),
    ("human", "질문: {question}\n\n문서: {document}")
])

score_chain = rerank_prompt | llm | StrOutputParser()


def llm_rerank(question: str, docs: list[Document], top_n: int = 3):
    """LLM으로 각 문서의 관련성 점수를 매기고 상위 top_n개 문서를 반환한다."""
    scored = []

    for doc in docs:
        try:
            score_text = score_chain.invoke({
                "question": question,
                "document": doc.page_content[:800]
            }).strip()
            score = int(score_text)
        except Exception:
            score = 0

        scored.append((score, doc))

    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:top_n]


question = "2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?"

base_retriever = vector_store.as_retriever(search_kwargs={"k": 8})
candidates = base_retriever.invoke(question)
reranked = llm_rerank(question, candidates, top_n=3)

print(f"LLM Reranking 결과: {len(reranked)}개")
for i, (score, doc) in enumerate(reranked, start=1):
    print(f"\n[{i}] 관련성 점수: {score} | source: {doc_label(doc)}")
    print(doc.page_content[:400])


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


LLM Reranking 결과: 3개

[1] 관련성 점수: 10 | source: chunk_id=21 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md
## **4.1 2025년 AI 산업·기술·정책의 트렌드 현황** 

- 앞 장에서는 2025년 주요 매체 데이터를 바탕으로 산업·기술·정책 전반에서 나타난 AI 담론의 구조적 흐름을 도출하였음 

- - 분석 결과를 통해 세 분야는 서로 다른 영역에서 출발하더라도 활용 확대 → 기술적 고도화 요구 증가 → 제도적 대응 심화 로 이어지는 연속적 흐름을 형성하고 있음을 확인하였음 

- 2026년에는 이 세 방향성이 더욱 맞물려 산업 구조의 재배치, 기술 체계의 정교화, 규제·책임 체계의 본격적 적용으로 이어지는 개략적 전망을 제시함 

## **4.2 2026년 AI 산업 전망** 

2026년 AI 시장은 선도 기업을 중심으로 내재화가 빠르게 진행 되며 시장 규모가 지속 확대될 전망 

- - AI 활용이

[2] 관련성 점수: 10 | source: chunk_id=23 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md
- - 고품질 데이터 생성·멀티모달·고급 추론 기술이 결합되며 AI의 상황 이해와 문제 해결 능력이 향상되고, 서비스·산업 전반의 활용도도 확대될 전망임 

- 기술 고도화에 따라 안전성·정합성 검증의 중요성이 커지면서, 신뢰성 평가와 표준화 마련이 AI 기술 확산의 기반 인프라로 자리 잡을 것으로 전망됨 

- - 합성데이터의 품질 편차, 추론형 AI의 환각(Hallucination), 멀티모달 모델 비교 기준 부재 등은 기술 적용 과정에서 안전성·정합성 검증 체계의 필요성을 강화하는 요소로 작용함 

- - 특히 데이터 품질 표준화와 모델 신뢰성 평가 기준 구축이 기능 고도화 기술의 산업 적용 확산을 뒷받침할 핵심 인프라로 요구됨 

4) https://watermanau

## Flashrank Reranking 전후 비교

같은 질문에 대해 단순 벡터 검색 결과와 Reranking 적용 결과를 비교한다.

확인할 점은 다음과 같다.

1. 벡터 검색 Top-K에 관련 없는 chunk가 섞이는지 확인한다.
2. Reranking 후 질문과 직접 관련 있는 문서가 상위로 올라오는지 확인한다.
3. 표나 특정 섹션 관련 질문에서 Reranking 효과가 있는지 확인한다.


In [4]:
question = "추론형 데이터는 어떤 유형으로 나뉘나요?"

# 벡터 검색 결과
# naive_results = vector_store.as_retriever(search_kwargs={"k": 5}).invoke(question)

# print("[벡터 검색 결과 - 기존 순서]")
# for i, doc in enumerate(naive_results, start=1):
#     print(f"\n[{i}] source: {doc_label(doc)}")
#     print(doc.page_content[:300].replace("\n", " "))
#------------------------------------------------------------------

# 벡터 검색 결과와 distance score를 함께 가져오기
naive_results_with_score = vector_store.similarity_search_with_score(
    question,
    k=10
)

print("[벡터 검색 결과 - 기존 순서]")
print("ChromaDB distance score: 낮을수록 질문과 더 유사함")
for i, (doc, score) in enumerate(naive_results_with_score, start=1):
    print(f"\n[{i}] distance={score:.4f} | source: {doc_label(doc)}")
    print(doc.page_content[:300])

# Reranking 적용 결과
reranked_results = compression_retriever.invoke(question)

print("\n[Flashrank Reranking 적용 후]")
for i, doc in enumerate(reranked_results, start=1):
    score = doc.metadata.get("relevance_score", "N/A")
    print(f"\n[{i}] relevance_score={score} | source: {doc_label(doc)}")
    print(doc.page_content[:300])


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[벡터 검색 결과 - 기존 순서]
ChromaDB distance score: 낮을수록 질문과 더 유사함

[1] distance=0.9631 | source: chunk_id=27 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md
## **< 추론형 데이터 종류(예시) >** 

|**구분**|**설명**|
|---|---|
|**① 단계적 '과정' 추론**|-
AI가 최종 결론에 도달하는 '과정' 자체를 학습하는 데 중점을 둔 데이터
- 절차적 추론 데이터(CoT), 상호작용적 추론 데이터 등
-**(예시)**민원처리 절차의 체크리스트 및 결정 기준|
|**② '근거' 기반 설명**|-
AI가 답을 말할 때 어떤 근거를 인용해야 하는지 학습시키기 위한 데이터
- 자연어 설명(NLE) 데이터, 추출적 근거 데이터 등
-**(예시)**정책 Q&A 공공데이터 개

[2] distance=1.2357 | source: chunk_id=12 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md
- - 해외 매체는 의미 왜곡을 최소화하고 분석 기준을 일관되게 유지하기 위해 한국어로 번역 후 요약하였으며, 이를 통해 분석 절차의 체계성과 비교 가능성을 확보함 

[ 그림 1 ] 분야별 데이터셋 구성 예시

## **2.2 연구 방법** 

- 본 분석은 R Studio 4.5.2 환경과 텍스트 분석 패키지를 활용해 MeCab 형태소 분석기 및 명사 토큰화 알고리즘으로 원문을 구조화함 

- - 전처리 과정에서는 숫자, 한 글자 단어, 조사, 불용어, 그리고 AI 문헌에서 빈번히 등장하는 단순 용어와 고유명사(국가명·기관명 등

[3] distance=1.2466 | source: chunk_id=26 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md
## **5.2 AI 기술 토픽으로 본 노력 및 대응*

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"



[Flashrank Reranking 적용 후]

[1] relevance_score=0.9996861815452576 | source: chunk_id=2 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md
- 「한국지능정보사회진흥원(NIA)」이라고 밝혀주시기 바랍니다. 

- ❸ 본 보고서의 내용은 한국지능정보사회진흥원(NIA)의 

- 공식 견해와 다를 수 있습니다. 

**작성  한국지능정보사회진흥원** 김시연 수석(siyeon@nia.or.kr, 053-230-4249) 추지혜 책임(chuu@nia.or.kr, 053-230-4247) 전민석 주임(msj1224@nia.or.kr, 053-230-4205) 

**기획  한국지능정보사회진흥원 인공지능데이터본부** 신신애 본부장(sashin@nia.or.kr, 053-230-420

[2] relevance_score=0.9996843934059143 | source: chunk_id=7 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md
- - 생성형 AI 고도화, AI 인프라 경쟁, 안전·책임성 강화, 국제 기준 정비 등 주요 변화 요인을 식별하여 산업·기술·정책 분야의 변화 방향성을 명확히 도출함 

- 더 나아가 분석 결과를 기반으로 국가 정책, 기업 전략, 기술 개발 방향 설정에 활용 가능한 실질적 인사이트를 제공하고, 중장기 전략 수립을 위한 근거를 제공하고자 함 

- - 특히 정책 담당자에게는 AI 기본법·규제체계 설계 등 법적 기반 마련 시 고려해야 할 우선순위를, 기업에는 투자·사업 포트폴리오 조정 방향을, 연구기관에는 후속 정량·정성 연구가 필요한

[3] relevance_score=0.9996470808982849 | source: chunk_id=13 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md
## 전처리

### 핵심 차이
| 구분                  | 코드                                | score 해석    |
| ------------------- | --------------------------------- | ----------- |
| 벡터 검색               | `similarity_search_with_score()`  | 낮을수록 유사     |
| Flashrank Reranking | `doc.metadata["relevance_score"]` | 높을수록 관련성 높음 |


## Reranking 포함 RAG 체인

Reranking을 적용한 검색 결과를 `context`로 사용하여 최종 답변을 생성한다.

실행 흐름은 다음과 같다.

```text
질문 입력
  ↓
벡터 검색으로 후보 문서 검색
  ↓
Reranker로 관련성 높은 문서 재선별
  ↓
선별된 문서를 context로 구성
  ↓
LLM 답변 생성
```


In [7]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


answer_prompt = ChatPromptTemplate.from_messages([
    ("system", """아래 문서에 근거해서만 질문에 답하세요.
문서에서 확인할 수 없는 내용은 '문서에서 확인할 수 없습니다'라고 답하세요.
한줄에 한문장씩 깔끔하게 정리하세요.

[문서]
{context}"""),
    ("human", "{question}")
])

rerank_rag_chain = (
    {
        "context": compression_retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | answer_prompt
    | llm
    | StrOutputParser()
)

answer = rerank_rag_chain.invoke(
    "AI 에이전트는 기업 운영 방식에 어떤 변화를 가져오나요?"
    # "2026년 AI 산업 전망에서 AI 에이전트는 기업 운영 방식에 어떤 변화를 가져올 것으로 전망되나요?"
)

print(answer)


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


AI 에이전트의 도입 확산은 업무 흐름과 운영 방식이 재구성되는 초기 징후로 볼 수 있습니다. AI 산업은 인프라 중심 운영 모델화가 동시에 나타나며, AI가 산업의 핵심 인프라로 자리 잡아가는 흐름을 시사합니다.


## 검색 품질 간단 평가

Reranking 전후 검색 결과가 질문과 얼마나 직접적으로 연결되는지 간단히 비교한다.  
정량 평가는 아니며, 수업용 관찰 평가이다.

In [6]:
test_questions = [
    "AI 산업, AI 기술, AI 정책의 핵심 이슈를 비교해줘.",
    "추론형 데이터는 어떤 유형으로 나뉘나요?",
    "AI 에이전트는 기업 운영 방식에 어떤 변화를 가져오나요?"
]

for q in test_questions:
    print("=" * 100)
    print(f"[질문] {q}")

    naive_docs = vector_store.as_retriever(search_kwargs={"k": 3}).invoke(q)
    rerank_docs = compression_retriever.invoke(q)

    print("\n[벡터 검색 Top-3]")
    for i, doc in enumerate(naive_docs, start=1):
        print(f"{i}. {doc_label(doc)}")

    print("\n[Reranking Top-3]")
    for i, doc in enumerate(rerank_docs, start=1):
        score = doc.metadata.get("relevance_score", "N/A")
        print(f"{i}. {doc_label(doc)} | relevance_score={score}")

[질문] AI 산업, AI 기술, AI 정책의 핵심 이슈를 비교해줘.


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"



[벡터 검색 Top-3]
1. chunk_id=6 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md
2. chunk_id=19 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md
3. chunk_id=20 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md

[Reranking Top-3]
1. chunk_id=7 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md | relevance_score=0.9996312260627747
2. chunk_id=24 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md | relevance_score=0.9996051788330078
3. chunk_id=25 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md | relevance_score=0.9995834231376648
[질문] 추론형 데이터는 어떤 유형으로 나뉘나요?


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"



[벡터 검색 Top-3]
1. chunk_id=27 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md
2. chunk_id=12 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md
3. chunk_id=26 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md

[Reranking Top-3]
1. chunk_id=2 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md | relevance_score=0.9996861815452576
2. chunk_id=7 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md | relevance_score=0.9996843934059143
3. chunk_id=13 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md | relevance_score=0.9996470808982849
[질문] AI 에이전트는 기업 운영 방식에 어떤 변화를 가져오나요?


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"



[벡터 검색 Top-3]
1. chunk_id=19 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md
2. chunk_id=22 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md
3. chunk_id=7 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md

[Reranking Top-3]
1. chunk_id=6 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md | relevance_score=0.9995517134666443
2. chunk_id=19 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md | relevance_score=0.9995115399360657
3. chunk_id=15 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md | relevance_score=0.999451220035553


## 정리

| 방법 | 장점 | 단점 |
|---|---|---|
| FlashrankRerank | 로컬에서 실행 가능, API 키 불필요, 빠른 편 | 모델 품질과 언어 처리 성능에 제한이 있을 수 있음 |
| LLM Reranking | 도메인 문맥을 유연하게 판단 가능, 별도 rerank 모델 불필요 | 후보 문서 수만큼 LLM 호출이 발생하여 느리고 비용이 듦 |
| Cohere Rerank | 실무에서 자주 고려되는 API 기반 reranker | 별도 API 키와 비용이 필요하며, 본 실습에서는 개념 소개만 다룸 |

**핵심**: 벡터 검색은 후보 문서를 빠르게 찾는 데 유리하고, Reranking은 그 후보 중 질문과 더 관련 있는 문서를 정밀하게 고르는 데 유리하다.

AI 트렌드 보고서처럼 산업·기술·정책 섹션이 서로 의미적으로 가까운 문서에서는 단순 벡터 검색만으로는 관련성이 낮은 chunk가 함께 검색될 수 있다.

따라서 Advanced RAG에서는 `1단계 벡터 검색 → 2단계 Reranking` 구조를 자주 사용하고, 재정렬된 상위 문서를 최종 context로 구성한다.

다음 단계: **Contextual Compression** → 선택된 문서 안에서 질문과 관련 있는 부분만 추출하거나 불필요한 내용을 줄이는 방법을 실습한다.


## 최신 Reranking 확장 방법

이번 실습에서는 FlashrankRerank와 LLM 기반 Reranking을 사용했다.  
실무에서는 다음과 같은 방식도 함께 고려할 수 있다.

| 방법 | 특징 | 적합한 상황 |
|---|---|---|
| Hugging Face Cross-Encoder Reranker | BGE, Qwen, GTE, mixedbread 계열 모델 사용 가능 | 로컬 또는 GPU 서버에서 직접 운영할 때 |
| Cohere Rerank | API 기반 상용 reranker | 품질과 구현 편의성이 중요할 때 |
| Jina Reranker | 다국어 검색, 긴 문서, 코드 관련 검색에 활용 가능한 reranker | 한국어·영어 혼합 문서나 다양한 형식의 문서 검색이 필요할 때 |
| Hybrid Search + Reranking | BM25와 Vector Search 후보를 통합한 뒤 재정렬 | 키워드와 의미 검색을 함께 강화할 때 |
| Listwise LLM Reranking | 여러 문서를 한 번에 비교하여 순위화 | 문서 간 상대 비교가 중요할 때 |
| Reranker 서버 운영 | vLLM 등으로 reranker를 별도 서버로 운영 | 운영 환경에서 성능과 확장성이 중요할 때 |

핵심은 하나의 Reranker가 항상 최선은 아니라는 점이다.  
문서 언어, 비용, 응답 속도, 운영 환경, 정확도 요구 수준에 따라 적절한 방식을 선택해야 한다.